<a href="https://colab.research.google.com/github/billydavisiv/BUAD-427/blob/main/BUAD427_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1
## Billy Davis

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/BUAD_427/Week_3/Election.csv'
df = pd.read_csv(file_path)

print(df.head())

  Gender    Age  Variant     Party  Donate  DonationAmt Newsletter Share  \
0      F  18-24        1  Democrat       0            0          N     N   
1      F  35-44        1  Democrat       0            0          N     N   
2      M    65+        2  Democrat       0            0          N     N   
3      M    65+        2  Democrat       0            0          N     N   
4      M  35-44        1  Democrat       0            0          N     N   

  ShareType  
0       NaN  
1       NaN  
2       NaN  
3       NaN  
4       NaN  


# 1. Background Analysis

## a. What is the main objective of this experiment? What is the experiment trying to learn or determine?

We want to test which variant resulted in the greatest amount of money donated and shares administered. This is because these are tied directly to financial support and exposure for the candidate, which increase the likelihood that they are voted for.

## b. What are the different variants being tested? What is the rationale or motivation behind these variations?

Variant 1 is a speficic reminder for people to cast a ballot. This is useful because, if administered to known members of the preferred party, will remind them to cast a vote for their candidate when the time comes. Variant 2 is an appeal to the voter's motivation and is inteded to bring them a sense of pride for their affiliation. It also dircetly ties their vote to the execution of their work and goals. Variant 3 is intended to create urgency. It puts the voters on edge and makes them think that the stakes are high, maybe even higher than they truly are.

## c. How does the experiment ensure random assignment and data collection? If you believe the randomization could be improved, identify the potential issues and suggest improvements.

The first assumption that the campaigners must make is that they are only able to sample emails that they have. Only using known emails introduces a bias and may exclude insights on new voters. Additionally, the collection of the 16,000 entrants would have to be completely random within the dataset, and the administration of the variants among that population would have to be equally randomized. If the process is truly random, we should see a fairly even spread of votes across all areas.

## d. A good sample should reasonably represent the target population, including important characteristics such as gender, age, or other relevant factors. How representative is the sample used in this experiment?

Unfortunately, we don't have as much characeristic data as we ideally should. We know the voter's age and gender, but we do not have any data on the needs and wants side. We do not known what motivates them and what would appeal to them (hence, of course, why we are A/B testing). We also do not have data on whether the person is a Republican or not, and, based on affiliation metrics, we could assume that many "Other" candidates are Republican, we cannot be certain. If we are trying to have non-Democrats flip their vote, then we should know more about where they come from.

## e. What are the potential limitations or issues with this experiment? How might they affect the validity or interpretation of the results?

One glaring issue that we do not know is whether or not they actually voted, nor do we know who they voted for if they did. If the campaigner's goals are to get a vote across the finish line, they have to have a system in place to monitor that finish line. They also do not have any data on referrals via sharing. This would be a helpful resource to see how important a share actually ends up being. In other words, if no one reacts to a shared message, then what value does it have?

# 2. Compute the percentage of supporters who pledge their support on social media for each of the three variants. Are there any variants statistically better than Variant 1 at a 2.5% level?

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# Calculate total supporters per variant
total_per_variant = df.groupby('Variant').size()

# Identify social media shares (Twitter or Facebook)
social_media_shares = df[
    (df['Share'] == 'Y') &
    ((df['ShareType'] == 'Twitter') | (df['ShareType'] == 'Facebook'))
]

# Count social media pledgers per variant
social_media_pledgers_per_variant = social_media_shares.groupby('Variant').size()

# Combine into a DataFrame for easier calculation and display
results_df = pd.DataFrame({
    'Total': total_per_variant,
    'SocialMediaPledgers': social_media_pledgers_per_variant
}).fillna(0) # Fill NaN with 0 for variants that might not have social media pledgers

# Compute percentage
results_df['Percentage'] = (results_df['SocialMediaPledgers'] / results_df['Total']) * 100

print("Percentage of supporters who pledged support on social media for each variant:")
print(results_df)

# Statistical test
alpha = 0.025 # 2.5% significance level for one-tailed test

# Variant 1 data
n1 = results_df.loc[1, 'Total']
x1 = results_df.loc[1, 'SocialMediaPledgers']

# Variant 2 data
n2 = results_df.loc[2, 'Total']
x2 = results_df.loc[2, 'SocialMediaPledgers']

# Variant 3 data
n3 = results_df.loc[3, 'Total']
x3 = results_df.loc[3, 'SocialMediaPledgers']

print("\n--- Statistical Comparison (Variant 2 vs Variant 1) ---")
# H0: p2 <= p1, H1: p2 > p1
# We want to test if the proportion of Variant 2 is LARGER than Variant 1
z_stat_v2_v1, p_value_v2_v1 = proportions_ztest(np.array([x2, x1]), np.array([n2, n1]), alternative='larger')

print(f"Variant 2 vs Variant 1: Z-statistic = {z_stat_v2_v1:.4f}, p-value = {p_value_v2_v1:.4f}")
if p_value_v2_v1 < alpha:
    print(f"Conclusion: Variant 2 is statistically better than Variant 1 at {alpha*100}% significance level.")
else:
    print(f"Conclusion: Variant 2 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")

print("\n--- Statistical Comparison (Variant 3 vs Variant 1) ---")
# H0: p3 <= p1, H1: p3 > p1
# We want to test if the proportion of Variant 3 is LARGER than Variant 1
z_stat_v3_v1, p_value_v3_v1 = proportions_ztest(np.array([x3, x1]), np.array([n3, n1]), alternative='larger')

print(f"Variant 3 vs Variant 1: Z-statistic = {z_stat_v3_v1:.4f}, p-value = {p_value_v3_v1:.4f}")
if p_value_v3_v1 < alpha:
    print(f"Conclusion: Variant 3 is statistically better than Variant 1 at {alpha*100}% significance level.")
else:
    print(f"Conclusion: Variant 3 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")

Percentage of supporters who pledged support on social media for each variant:
         Total  SocialMediaPledgers  Percentage
Variant                                        
1         5362                  129    2.405819
2         5322                   94    1.766253
3         5316                  116    2.182092

--- Statistical Comparison (Variant 2 vs Variant 1) ---
Variant 2 vs Variant 1: Z-statistic = -2.3121, p-value = 0.9896
Conclusion: Variant 2 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Statistical Comparison (Variant 3 vs Variant 1) ---
Variant 3 vs Variant 1: Z-statistic = -0.7720, p-value = 0.7800
Conclusion: Variant 3 is NOT statistically better than Variant 1 at 2.5% significance level.


No, neither performed better than variant 1 with regards to social media shares. Variant 3 was close behind, and Variant 2 was signficantly lower.



# 3. Repeat the computations from Question 1 but conduct the analysis separately for each party affiliation. Are any variants statistically better than Variant 1 at the 2.5% significance level? If so, which variants outperform Variant 1 for which parties, and by how much? Do you think the potential increase in sharing is meaningful for an election campaign? Briefly explain.

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

alpha = 0.025 # 2.5% significance level for one-tailed test

for party in df['Party'].unique():
    print(f"\n--- Analysis for Party: {party} ---")
    party_df = df[df['Party'] == party]

    # Calculate total supporters per variant for this party
    total_per_variant_party = party_df.groupby('Variant').size()

    # Identify social media shares for this party
    social_media_shares_party = party_df[
        (party_df['Share'] == 'Y') &
        ((party_df['ShareType'] == 'Twitter') | (party_df['ShareType'] == 'Facebook'))
    ]

    # Count social media pledgers per variant for this party
    social_media_pledgers_per_variant_party = social_media_shares_party.groupby('Variant').size()

    # Combine into a DataFrame for easier calculation and display
    results_df_party = pd.DataFrame({
        'Total': total_per_variant_party,
        'SocialMediaPledgers': social_media_pledgers_per_variant_party
    }).fillna(0) # Fill NaN with 0 for variants that might not have social media pledgers

    # Ensure all variants (1, 2, 3) are present, even if no shares/totals
    for v in [1, 2, 3]:
        if v not in results_df_party.index:
            results_df_party.loc[v] = {'Total': 0, 'SocialMediaPledgers': 0}
    results_df_party = results_df_party.sort_index()

    # Compute percentage
    results_df_party['Percentage'] = (results_df_party['SocialMediaPledgers'] / results_df_party['Total']) * 100
    results_df_party['Percentage'] = results_df_party['Percentage'].fillna(0) # Handle division by zero if Total is 0

    print("Percentage of supporters who pledged support on social media for each variant:")
    print(results_df_party)

    # Variant 1 data for this party
    n1_party = results_df_party.loc[1, 'Total']
    x1_party = results_df_party.loc[1, 'SocialMediaPledgers']

    # Perform statistical tests only if Variant 1 has data
    if n1_party > 0:
        # Variant 2 data for this party
        n2_party = results_df_party.loc[2, 'Total']
        x2_party = results_df_party.loc[2, 'SocialMediaPledgers']

        # Variant 3 data for this party
        n3_party = results_df_party.loc[3, 'Total']
        x3_party = results_df_party.loc[3, 'SocialMediaPledgers']

        print("\n--- Statistical Comparison (Variant 2 vs Variant 1) ---")
        # Test if Variant 2 is LARGER than Variant 1
        if n2_party > 0 and n1_party > 0: # Ensure enough data for comparison
            z_stat_v2_v1, p_value_v2_v1 = proportions_ztest(np.array([x2_party, x1_party]), np.array([n2_party, n1_party]), alternative='larger')
            print(f"Variant 2 vs Variant 1: Z-statistic = {z_stat_v2_v1:.4f}, p-value = {p_value_v2_v1:.4f}")
            if p_value_v2_v1 < alpha:
                print(f"Conclusion: Variant 2 is statistically better than Variant 1 at {alpha*100}% significance level.")
            else:
                print(f"Conclusion: Variant 2 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough data to compare Variant 2 and Variant 1.")

        print("\n--- Statistical Comparison (Variant 3 vs Variant 1) ---")
        # Test if Variant 3 is LARGER than Variant 1
        if n3_party > 0 and n1_party > 0: # Ensure enough data for comparison
            z_stat_v3_v1, p_value_v3_v1 = proportions_ztest(np.array([x3_party, x1_party]), np.array([n3_party, n1_party]), alternative='larger')
            print(f"Variant 3 vs Variant 1: Z-statistic = {z_stat_v3_v1:.4f}, p-value = {p_value_v3_v1:.4f}")
            if p_value_v3_v1 < alpha:
                print(f"Conclusion: Variant 3 is statistically better than Variant 1 at {alpha*100}% significance level.")
            else:
                print(f"Conclusion: Variant 3 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough data to compare Variant 3 and Variant 1.")
    else:
        print("Variant 1 has no data for this party, skipping statistical comparisons.")


--- Analysis for Party: Democrat ---
Percentage of supporters who pledged support on social media for each variant:
         Total  SocialMediaPledgers  Percentage
Variant                                        
1         3816                   98    2.568134
2         3726                   61    1.637144
3         3662                   89    2.430366

--- Statistical Comparison (Variant 2 vs Variant 1) ---
Variant 2 vs Variant 1: Z-statistic = -2.8138, p-value = 0.9976
Conclusion: Variant 2 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Statistical Comparison (Variant 3 vs Variant 1) ---
Variant 3 vs Variant 1: Z-statistic = -0.3814, p-value = 0.6486
Conclusion: Variant 3 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Analysis for Party: Other ---
Percentage of supporters who pledged support on social media for each variant:
         Total  SocialMediaPledgers  Percentage
Variant                                        
1    

Even in light of this deeper analysis, we do not see any statistically significant differences at a 2.5% level. It is worth noting, however, that Variant 3 for Other was over double that of Variant 1, so, while not statistically signficant, it is worth considering or re-testing.

Testing for shares is a good signal on whether someone will vote for a candidate, but there is a nuance. We can say, with reasonable confidence, that someone who shares will vote for our candidate, but we cannot reasonably say that someone who does not share will not vote for the candidate. It's a one-sided measure, but it is, nevertheless, useful. It also could inform us on where our voters congregate digitally, which can educate where they post advertisements.

# 4. Think carefully about what you’ve done in Question 1 and Question 2. Is party a confounding factor? If yes, what does that tell you based on AB tests.

Absolutely! We are comparing people who have been sold on the idea and people who have not. Party affiliation is a strong enough factor in the data that we should always see less variance with those directky affiliated with the campaigning party. We don't have to sell them as easily. Independents or other voters, however, may be on the fence or opposed to the campaign's political angle, so this is where an A/B test would naturally shine more.

All of this leads us to seeing that someone in a party would be more likely to advocate for someone in their party than someone who is not affiliated. It is a confounding factor.

# 5. Compute the probability a supporter donates under each variant. Do your analysis separately for each party. Are any variants statistically (at 2.5%) better than Variant 1? If so, which variants outperform Variant 1 for which parties, and by how much? Do you think the potential increase in donate likelihood is meaningful for an election campaign? Briefly explain.

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

alpha = 0.025 # 2.5% significance level for one-tailed test

for party in df['Party'].unique():
    print(f"\n--- Analysis for Party: {party} ---")
    party_df = df[df['Party'] == party]

    # Calculate total supporters per variant for this party
    total_per_variant_party = party_df.groupby('Variant').size()

    # Count donators per variant for this party
    donators_per_variant_party = party_df[party_df['Donate'] == 1].groupby('Variant').size()

    # Combine into a DataFrame for easier calculation and display
    results_df_party_donations = pd.DataFrame({
        'Total': total_per_variant_party,
        'Donators': donators_per_variant_party
    }).fillna(0) # Fill NaN with 0 for variants that might not have donators

    # Ensure all variants (1, 2, 3) are present, even if no donations/totals
    for v in [1, 2, 3]:
        if v not in results_df_party_donations.index:
            results_df_party_donations.loc[v] = {'Total': 0, 'Donators': 0}
    results_df_party_donations = results_df_party_donations.sort_index()

    # Compute percentage
    results_df_party_donations['Donation_Probability'] = (results_df_party_donations['Donators'] / results_df_party_donations['Total']) * 100
    results_df_party_donations['Donation_Probability'] = results_df_party_donations['Donation_Probability'].fillna(0) # Handle division by zero if Total is 0

    print("Probability a supporter donates for each variant:")
    print(results_df_party_donations)

    # Variant 1 data for this party
    n1_party = results_df_party_donations.loc[1, 'Total']
    x1_party = results_df_party_donations.loc[1, 'Donators']
    p1_party = results_df_party_donations.loc[1, 'Donation_Probability']

    # Perform statistical tests only if Variant 1 has data (or reasonable sample size)
    # A minimum of 5 successful and 5 unsuccessful outcomes is often recommended for z-test
    if n1_party > 0 and x1_party >= 5 and (n1_party - x1_party) >= 5:
        # Variant 2 data for this party
        n2_party = results_df_party_donations.loc[2, 'Total']
        x2_party = results_df_party_donations.loc[2, 'Donators']
        p2_party = results_df_party_donations.loc[2, 'Donation_Probability']

        # Variant 3 data for this party
        n3_party = results_df_party_donations.loc[3, 'Total']
        x3_party = results_df_party_donations.loc[3, 'Donators']
        p3_party = results_df_party_donations.loc[3, 'Donation_Probability']

        print("\n--- Statistical Comparison (Variant 2 vs Variant 1) ---")
        # Test if Variant 2 is LARGER than Variant 1
        if n2_party > 0 and x2_party >= 5 and (n2_party - x2_party) >= 5:
            z_stat_v2_v1, p_value_v2_v1 = proportions_ztest(np.array([x2_party, x1_party]), np.array([n2_party, n1_party]), alternative='larger')
            print(f"Variant 2 vs Variant 1: Z-statistic = {z_stat_v2_v1:.4f}, p-value = {p_value_v2_v1:.4f}")
            if p_value_v2_v1 < alpha:
                print(f"Conclusion: Variant 2 is statistically better than Variant 1 at {alpha*100}% significance level.")
                print(f"Outperformance: {p2_party - p1_party:.2f}% percentage points.")
            else:
                print(f"Conclusion: Variant 2 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough data for Variant 2 to conduct a reliable comparison.")

        print("\n--- Statistical Comparison (Variant 3 vs Variant 1) ---")
        # Test if Variant 3 is LARGER than Variant 1
        if n3_party > 0 and x3_party >= 5 and (n3_party - x3_party) >= 5:
            z_stat_v3_v1, p_value_v3_v1 = proportions_ztest(np.array([x3_party, x1_party]), np.array([n3_party, n1_party]), alternative='larger')
            print(f"Variant 3 vs Variant 1: Z-statistic = {z_stat_v3_v1:.4f}, p-value = {p_value_v3_v1:.4f}")
            if p_value_v3_v1 < alpha:
                print(f"Conclusion: Variant 3 is statistically better than Variant 1 at {alpha*100}% significance level.")
                print(f"Outperformance: {p3_party - p1_party:.2f}% percentage points.")
            else:
                print(f"Conclusion: Variant 3 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough data for Variant 3 to conduct a reliable comparison.")
    else:
        print("Not enough data for Variant 1 (or its variants) in this party to conduct reliable statistical comparisons (need at least 5 successes and 5 failures).")


--- Analysis for Party: Democrat ---
Probability a supporter donates for each variant:
         Total  Donators  Donation_Probability
Variant                                       
1         3816       211              5.529350
2         3726       262              7.031669
3         3662       174              4.751502

--- Statistical Comparison (Variant 2 vs Variant 1) ---
Variant 2 vs Variant 1: Z-statistic = 2.6904, p-value = 0.0036
Conclusion: Variant 2 is statistically better than Variant 1 at 2.5% significance level.
Outperformance: 1.50% percentage points.

--- Statistical Comparison (Variant 3 vs Variant 1) ---
Variant 3 vs Variant 1: Z-statistic = -1.5216, p-value = 0.9359
Conclusion: Variant 3 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Analysis for Party: Other ---
Probability a supporter donates for each variant:
         Total  Donators  Donation_Probability
Variant                                       
1          494        23          

Upon testing, the only statistically significant varaint was Variant 2 for Democrats, which showed an increase of 1.5 percentage points. It's great that this is statistically significant, but when you actually consider the relative change in a full 0%-100% scale, a jump from 5.5% to 7% is not a massive leap. However, spreading this out across the entire voter base in the U.S. could prove to be a meaningful change. That 1.5% margin could be millions of dollars worth of donations if the data has given us an accurate preview.

# 6. Compute the average amount a supporter donates under each variant. Do your analysis separately for each party. Are any variants statistically (at 2.5%) better than variant 1? If so, which variants outperform Variant 1 for which parties, and by how much? Do you think the potential increase in average donate amount is meaningful for an election campaign? Briefly explain.

In [ ]:
import pandas as pd
from scipy import stats
import numpy as np

alpha = 0.025 # 2.5% significance level for one-tailed test

for party in df['Party'].unique():
    print(f"\n--- Analysis for Party: {party} ---")
    party_df = df[df['Party'] == party]

    # Filter for actual donators (DonationAmt > 0)
    donors_party_df = party_df[party_df['DonationAmt'] > 0]

    # Calculate average donation amount per variant for this party
    avg_donation_per_variant_party = donors_party_df.groupby('Variant')['DonationAmt'].mean()
    count_donors_per_variant_party = donors_party_df.groupby('Variant')['DonationAmt'].count()
    total_donated_per_variant_party = donors_party_df.groupby('Variant')['DonationAmt'].sum()

    results_df_avg_donations = pd.DataFrame({
        'Average_Donation_Amt': avg_donation_per_variant_party,
        'Number_of_Donors': count_donors_per_variant_party,
        'Total_Donated_Amt': total_donated_per_variant_party
    }).fillna(0) # Fill NaN with 0 for variants that might not have donators

    # Ensure all variants (1, 2, 3) are present, even if no donors
    for v in [1, 2, 3]:
        if v not in results_df_avg_donations.index:
            results_df_avg_donations.loc[v] = {'Average_Donation_Amt': 0, 'Number_of_Donors': 0, 'Total_Donated_Amt': 0}
    results_df_avg_donations = results_df_avg_donations.sort_index()

    print("Average donation amount per donor for each variant:")
    print(results_df_avg_donations)

    # Get donation amounts for Variant 1, 2, and 3
    v1_donations = donors_party_df[donors_party_df['Variant'] == 1]['DonationAmt']
    v2_donations = donors_party_df[donors_party_df['Variant'] == 2]['DonationAmt']
    v3_donations = donors_party_df[donors_party_df['Variant'] == 3]['DonationAmt']

    # Variant 1 average for comparison
    avg_v1 = results_df_avg_donations.loc[1, 'Average_Donation_Amt']

    # Perform statistical tests only if Variant 1 has enough donors
    if len(v1_donations) >= 5: # Small sample size can make t-test unreliable

        print("\n--- Statistical Comparison (Variant 2 vs Variant 1) ---")
        if len(v2_donations) >= 5:
            # Perform independent t-test (Welch's t-test, no assumption of equal variance)
            t_stat_v2_v1, p_value_v2_v1_two_tailed = stats.ttest_ind(v2_donations, v1_donations, equal_var=False)

            # For one-tailed test (V2 > V1)
            if t_stat_v2_v1 > 0:
                p_value_v2_v1 = p_value_v2_v1_two_tailed / 2
            else:
                p_value_v2_v1 = 1 - (p_value_v2_v1_two_tailed / 2)

            print(f"Variant 2 vs Variant 1: T-statistic = {t_stat_v2_v1:.4f}, p-value (one-tailed) = {p_value_v2_v1:.4f}")
            if p_value_v2_v1 < alpha:
                print(f"Conclusion: Variant 2 is statistically better than Variant 1 at {alpha*100}% significance level.")
                outperformance_v2 = results_df_avg_donations.loc[2, 'Average_Donation_Amt'] - avg_v1
                print(f"Outperformance: ${outperformance_v2:.2f} increase in average donation amount.")
            else:
                print(f"Conclusion: Variant 2 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough donors for Variant 2 to conduct a reliable comparison (need at least 5 donors).")

        print("\n--- Statistical Comparison (Variant 3 vs Variant 1) ---")
        if len(v3_donations) >= 5:
            t_stat_v3_v1, p_value_v3_v1_two_tailed = stats.ttest_ind(v3_donations, v1_donations, equal_var=False)

            # For one-tailed test (V3 > V1)
            if t_stat_v3_v1 > 0:
                p_value_v3_v1 = p_value_v3_v1_two_tailed / 2
            else:
                p_value_v3_v1 = 1 - (p_value_v3_v1_two_tailed / 2)

            print(f"Variant 3 vs Variant 1: T-statistic = {t_stat_v3_v1:.4f}, p-value (one-tailed) = {p_value_v3_v1:.4f}")
            if p_value_v3_v1 < alpha:
                print(f"Conclusion: Variant 3 is statistically better than Variant 1 at {alpha*100}% significance level.")
                outperformance_v3 = results_df_avg_donations.loc[3, 'Average_Donation_Amt'] - avg_v1
                print(f"Outperformance: ${outperformance_v3:.2f} increase in average donation amount.")
            else:
                print(f"Conclusion: Variant 3 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough donors for Variant 3 to conduct a reliable comparison (need at least 5 donors).")
    else:
        print("Not enough donors for Variant 1 in this party to conduct reliable statistical comparisons (need at least 5 donors).")


--- Analysis for Party: Democrat ---
Average donation amount per donor for each variant:
         Average_Donation_Amt  Number_of_Donors  Total_Donated_Amt
Variant                                                           
1                   20.900474               211               4410
2                   21.221374               262               5560
3                   37.218391               174               6476

--- Statistical Comparison (Variant 2 vs Variant 1) ---
Variant 2 vs Variant 1: T-statistic = 0.7852, p-value (one-tailed) = 0.2164
Conclusion: Variant 2 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Statistical Comparison (Variant 3 vs Variant 1) ---
Variant 3 vs Variant 1: T-statistic = 28.0484, p-value (one-tailed) = 0.0000
Conclusion: Variant 3 is statistically better than Variant 1 at 2.5% significance level.
Outperformance: $16.32 increase in average donation amount.

--- Analysis for Party: Other ---
Average donation amount per don

This test revealed one area where we can see a statistically significant impact of a change, and that comes from Variant 3 among Democrats, with a $16.32 increase in donation amount.

This makes a huge difference in an election campaign, since, considering a visible number of Democrats donate, any increase would be directly tied to more money for further campaigning. More money is never bad here, and having it be that significant of an increase is promising.

# 7. Compute the probability a supporter signs up for the newsletter under each variant. Do your analysis separately for each party. Are any variants statistically (2.5%) better than variant 1? If so, which variants outperform Variant 1 for which parties, and by how much? Do you think the potential increase in donate likelihood is meaningful for an election campaign? Briefly explain.

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

alpha = 0.025 # 2.5% significance level for one-tailed test

print("### Newsletter Sign-up Probability Analysis ###\n")

for party in df['Party'].unique():
    print(f"\n--- Analysis for Party: {party} ---")
    party_df = df[df['Party'] == party]

    # Calculate total supporters per variant for this party
    total_per_variant_party = party_df.groupby('Variant').size()

    # Count newsletter sign-ups per variant for this party
    newsletter_signups_per_variant_party = party_df[party_df['Newsletter'] == 'Y'].groupby('Variant').size()

    # Combine into a DataFrame for easier calculation and display
    results_df_party_newsletter = pd.DataFrame({
        'Total': total_per_variant_party,
        'Newsletter_Signups': newsletter_signups_per_variant_party
    }).fillna(0) # Fill NaN with 0 for variants that might not have sign-ups

    # Ensure all variants (1, 2, 3) are present, even if no sign-ups/totals
    for v in [1, 2, 3]:
        if v not in results_df_party_newsletter.index:
            results_df_party_newsletter.loc[v] = {'Total': 0, 'Newsletter_Signups': 0}
    results_df_party_newsletter = results_df_party_newsletter.sort_index()

    # Compute percentage
    results_df_party_newsletter['Newsletter_Probability'] = (results_df_party_newsletter['Newsletter_Signups'] / results_df_party_newsletter['Total']) * 100
    results_df_party_newsletter['Newsletter_Probability'] = results_df_party_newsletter['Newsletter_Probability'].fillna(0) # Handle division by zero if Total is 0

    print("Probability a supporter signs up for the newsletter for each variant:")
    print(results_df_party_newsletter)

    # Variant 1 data for this party
    n1_party = results_df_party_newsletter.loc[1, 'Total']
    x1_party = results_df_party_newsletter.loc[1, 'Newsletter_Signups']
    p1_party = results_df_party_newsletter.loc[1, 'Newsletter_Probability']

    # Perform statistical tests only if Variant 1 has enough data (at least 5 successes and 5 failures)
    if n1_party > 0 and x1_party >= 5 and (n1_party - x1_party) >= 5:
        # Variant 2 data for this party
        n2_party = results_df_party_newsletter.loc[2, 'Total']
        x2_party = results_df_party_newsletter.loc[2, 'Newsletter_Signups']
        p2_party = results_df_party_newsletter.loc[2, 'Newsletter_Probability']

        # Variant 3 data for this party
        n3_party = results_df_party_newsletter.loc[3, 'Total']
        x3_party = results_df_party_newsletter.loc[3, 'Newsletter_Signups']
        p3_party = results_df_party_newsletter.loc[3, 'Newsletter_Probability']

        print("\n--- Statistical Comparison (Variant 2 vs Variant 1) ---")
        # Test if Variant 2 is LARGER than Variant 1
        if n2_party > 0 and x2_party >= 5 and (n2_party - x2_party) >= 5:
            z_stat_v2_v1, p_value_v2_v1 = proportions_ztest(np.array([x2_party, x1_party]), np.array([n2_party, n1_party]), alternative='larger')
            print(f"Variant 2 vs Variant 1: Z-statistic = {z_stat_v2_v1:.4f}, p-value = {p_value_v2_v1:.4f}")
            if p_value_v2_v1 < alpha:
                print(f"Conclusion: Variant 2 is statistically better than Variant 1 at {alpha*100}% significance level.")
                print(f"Outperformance: {p2_party - p1_party:.2f} percentage points.")
            else:
                print(f"Conclusion: Variant 2 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough data for Variant 2 to conduct a reliable comparison (need at least 5 sign-ups and 5 non-sign-ups).")

        print("\n--- Statistical Comparison (Variant 3 vs Variant 1) ---")
        # Test if Variant 3 is LARGER than Variant 1
        if n3_party > 0 and x3_party >= 5 and (n3_party - x3_party) >= 5:
            z_stat_v3_v1, p_value_v3_v1 = proportions_ztest(np.array([x3_party, x1_party]), np.array([n3_party, n1_party]), alternative='larger')
            print(f"Variant 3 vs Variant 1: Z-statistic = {z_stat_v3_v1:.4f}, p-value = {p_value_v3_v1:.4f}")
            if p_value_v3_v1 < alpha:
                print(f"Conclusion: Variant 3 is statistically better than Variant 1 at {alpha*100}% significance level.")
                print(f"Outperformance: {p3_party - p1_party:.2f} percentage points.")
            else:
                print(f"Conclusion: Variant 3 is NOT statistically better than Variant 1 at {alpha*100}% significance level.")
        else:
            print("Not enough data for Variant 3 to conduct a reliable comparison (need at least 5 sign-ups and 5 non-sign-ups).")
    else:
        print("Not enough data for Variant 1 (or its variants) in this party to conduct reliable statistical comparisons (need at least 5 sign-ups and 5 non-sign-ups).")

print("\n--- Summary of Newsletter Sign-up Analysis ---")
print("For Democrats: No variants were statistically better than Variant 1 in terms of newsletter sign-up probability.")
print("For Others: No variants were statistically better than Variant 1 in terms of newsletter sign-up probability.")
print("For Independents: No variants were statistically better than Variant 1 in terms of newsletter sign-up probability.")
print("\nIn this analysis, none of the variants significantly increased the probability of a supporter signing up for the newsletter compared to Variant 1 at a 2.5% significance level, for any of the party affiliations. The lack of significant difference here suggests that the current variants might not be optimized for driving newsletter sign-ups. The potential increase in donate likelihood refers to the donation probability or average donation amount, which was analyzed in previous questions. For newsletter sign-ups, an increase would generally be meaningful as it allows for continued communication and engagement with supporters, potentially leading to future donations or actions, even if no statistically significant increase was found in this specific test.")

### Newsletter Sign-up Probability Analysis ###


--- Analysis for Party: Democrat ---
Probability a supporter signs up for the newsletter for each variant:
         Total  Newsletter_Signups  Newsletter_Probability
Variant                                                   
1         3816                 125                3.275681
2         3726                  58                1.556629
3         3662                  53                1.447297

--- Statistical Comparison (Variant 2 vs Variant 1) ---
Variant 2 vs Variant 1: Z-statistic = -4.8509, p-value = 1.0000
Conclusion: Variant 2 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Statistical Comparison (Variant 3 vs Variant 1) ---
Variant 3 vs Variant 1: Z-statistic = -5.1850, p-value = 1.0000
Conclusion: Variant 3 is NOT statistically better than Variant 1 at 2.5% significance level.

--- Analysis for Party: Other ---
Probability a supporter signs up for the newsletter for each variant:
         Total 

The test did not show any statistically significant changes across variants for newsletter signups, but that does not detract from its importance. Newsletter signups do serve as an indication of loyalty and interest, so it could be cross-checked with shares to identify users who we can confidently assume will be on our side. Sending further donation and campaign-related emails their way could result in more money and exposure for the campaign, and that matters.

One area, however, where this data is unreliable, is for existing Democrats. They may already be signed up for newsletters, so their reported numbers may not be reliable under most circumstances.

#8. Compare all of your answers above. Which variant would you recommend for each party, and why? Are there potential issues with using different email subject headlines for voters from different parties? Briefly explain.

My choice for what to use would vary based on the party affiliation.

Based on what I've seen and what Gemini gathered, Variant 3 seemed to perform best amongst Democrats and Other, and this is backed by the massivce jump in donation amount in both. The massive $16.32 dollar jump observed with Democrats for Variant 3 is the stringest argument for Democrats. The smaller but still strong jump of 5.61 dollars when compared to Variant 1 for Other means that the campaign will have more funding to work with.

This differs from independents, who showed no significant change across any categories compared to Variant 1 at 2.5% significance. Therefore, making an alteration would simply not be worth the money.